In [1]:
# Standard setup cell - run before other cells.
import importlib.util
import subprocess
import sys
from pathlib import Path

def ensure_dependencies():
    """Install only packages used by this module."""
    required = {"numpy": "numpy", "matplotlib": "matplotlib", "pandas": "pandas"}
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_module_dir(start):
    """Find the shared membership module."""
    for root in [start, *start.parents]:
        for candidate in (root, root / "Fuzzy Logic"):
            if (candidate / "membership.py").exists():
                return candidate
        if (root / ".git").exists():
            break
    return None

ensure_dependencies()
module_dir = find_module_dir(Path.cwd())
if module_dir is None and "google.colab" in sys.modules:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git", str(clone_dir)], check=True)
    module_dir = find_module_dir(clone_dir)
if module_dir is None:
    raise RuntimeError("membership.py not found; open this notebook inside the repository")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from membership import linear_up, linear_down, triangular, trapezoidal

print("Python:", sys.version.split()[0])
print("Membership module:", module_dir / "membership.py")

Python: 3.12.4
Membership module: c:\Users\LEGION\Praktikum-KK\fuzzy-logic\membership.py


# Soal 1 - Pengaturan Kecepatan Kipas Sugeno Orde Nol

## 1. Perhitungan Manual untuk T = 28 C

### a. Derajat Keanggotaan Input T = 28 C
sejuk = linear_down(28, 20, 30) = (30 - 28) / (30 - 20) = 2 / 10 = 0.20
nyaman = triangular(28, 20, 30, 40) = (28 - 20) / (30 - 20) = 8 / 10 = 0.80
panas = linear_up(28, 30, 40) = 0.00

### b. Kekuatan Aktivasi wi dan Evaluasi Konsekuen zi
R1: IF suhu Sejuk THEN z1 = 20 persen -> w1 = 0.20, z1 = 20
R2: IF suhu Nyaman THEN z2 = 50 persen -> w2 = 0.80, z2 = 50
R3: IF suhu Panas THEN z3 = 90 persen -> w3 = 0.00, z3 = 90

### c. Rata Rata Berbobot
sum(wi) = 0.20 + 0.80 + 0.00 = 1.00
sum(wi * zi) = (0.20 * 20) + (0.80 * 50) + (0.00 * 90) = 4.0 + 40.0 + 0 = 44.0
output = 44.0 / 1.00 = 44.00 persen

---

## 2. Perhitungan Manual untuk Pembanding T = 36 C

### a. Derajat Keanggotaan Input T = 36 C
sejuk = linear_down(36, 20, 30) = 0.00
nyaman = triangular(36, 20, 30, 40) = (40 - 36) / (40 - 30) = 4 / 10 = 0.40
panas = linear_up(36, 30, 40) = (36 - 30) / (40 - 30) = 6 / 10 = 0.60

### b. Kekuatan Aktivasi wi dan Evaluasi Konsekuen zi
R1: IF suhu Sejuk THEN z1 = 20 persen -> w1 = 0.00, z1 = 20
R2: IF suhu Nyaman THEN z2 = 50 persen -> w2 = 0.40, z2 = 50
R3: IF suhu Panas THEN z3 = 90 persen -> w3 = 0.60, z3 = 90

### c. Rata Rata Berbobot
sum(wi) = 0.00 + 0.40 + 0.60 = 1.00
sum(wi * zi) = (0.00 * 20) + (0.40 * 50) + (0.60 * 90) = 0 + 20.0 + 54.0 = 74.0
output = 74.0 / 1.00 = 74.00 persen

---

## 3. Tabel Perbandingan dan Analisis

| Suhu | w1 Sejuk | w2 Nyaman | w3 Panas | Output Kipas | Aturan Aktif | Rentang Konsekuen Aktif |
| --- | --- | --- | --- | --- | --- | --- |
| 28 C | 0.20 | 0.80 | 0.00 | 44.00 persen | R1, R2 | 20 sampai 50 |
| 36 C | 0.00 | 0.40 | 0.60 | 74.00 persen | R2, R3 | 50 sampai 90 |

### Pemeriksaan Output dan Analisis Perubahan
pemeriksaan output menunjukkan bahwa output untuk 28 C yaitu 44.00 persen berada di antara konsekuen aturan aktif 20 sampai 50 dan output untuk 36 C yaitu 74.00 persen berada di antara 50 sampai 90
perubahan output terjadi karena saat suhu naik dari 28 C ke 36 C dominasi aturan aktif bergeser dari sejuk nyaman menjadi nyaman panas sehingga kecepatan kipas meningkat dari 44 persen menjadi 74 persen
kurva fungsi keanggotaan output tidak diperlukan pada metode sugeno karena nilai konsekuen setiap aturan sudah berupa konstanta atau persamaan linier bukan himpunan fuzzy sehingga penentuan nilai tegas dapat langsung dihitung menggunakan rata rata berbobot tanpa melalui proses defuzzifikasi berbasis luas area

In [2]:
# Sel 2 (Code)

# Definisikan fungsi weighted_average langsung di sini biar gak error NameError
def weighted_average(w, z):
    return np.sum(w * z) / np.sum(w)

def fuzz_temp(t):
    return {
        "sejuk": linear_down(t, 20, 30),
        "nyaman": triangular(t, 20, 30, 40),
        "panas": linear_up(t, 30, 40),
    }

rules_s1 = [
    {"rule": "R1", "in": "sejuk", "z": 20.0},
    {"rule": "R2", "in": "nyaman", "z": 50.0},
    {"rule": "R3", "in": "panas", "z": 90.0},
]

def eval_s1(t):
    deg = fuzz_temp(t)
    w = np.array([deg[r["in"]] for r in rules_s1])
    z = np.array([r["z"] for r in rules_s1])
    out = weighted_average(w, z)
    act = w > 0
    assert z[act].min() <= out <= z[act].max()
    return deg, w, z, out

for temp in [28, 36]:
    deg, w, z, out = eval_s1(temp)
    df = pd.DataFrame(
        {"rule": [r["rule"] for r in rules_s1], "w_i": w, "z_i": z, "w_i*z_i": w * z}
    )
    print(f"suhu {temp}°C")
    display(df.round(4))
    print(
        f"sum_w: {w.sum():.2f}, sum_wz: {(w*z).sum():.2f}, output: {out:.2f}%\n"
    )

suhu 28°C


,rule,w_i,z_i,w_i*z_i
0,R1,0.2,20.0,4.0
1,R2,0.8,50.0,40.0
2,R3,0.0,90.0,0.0


sum_w: 1.00, sum_wz: 44.00, output: 44.00%

suhu 36°C


,rule,w_i,z_i,w_i*z_i
0,R1,0.0,20.0,0.0
1,R2,0.4,50.0,20.0
2,R3,0.6,90.0,54.0


sum_w: 1.00, sum_wz: 74.00, output: 74.00%

